# GNN-Pruning — overnight single-seed sweep (Colab / A100)

Runs the full pruning matrix on CUDA. The large undirected graphs (Reddit,
ogbn-products, Yelp) use a **sparse-adjacency (SpMM)** path so they fit in
GPU memory; everything else runs full-batch dense. Wanda scores are **exact**
(the sparse path is numerically identical to the dense one — regression-tested).

**Before running:**
1. `Runtime → Change runtime type → A100 GPU` (Colab Pro). L4 (24 GB) also works
   for everything except possibly ogbn-products.
2. Colab Pro: enable **background execution** so the sweep survives tab close.
3. Run the cells top to bottom. The sweep is **idempotent** — if you disconnect,
   just re-run the setup + sweep cells and it resumes (finished cells are skipped).

**Known infeasible cell:** `reddit / gat`. Full-batch GAT on 23M edges needs
~188 GB (attention is inherently per-edge); it will OOM and be logged as a
failure in `run.log`. Reddit is still covered by GCN and GraphSAGE. Everything
else is expected to complete.


## 1. Confirm GPU

In [1]:
import torch
assert torch.cuda.is_available(), "No CUDA GPU — set Runtime → A100 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("torch:", torch.__version__)

GPU: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.1
torch: 2.11.0+cu128


## 2. Mount Drive (so results + datasets survive disconnects)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/gnn-pruning'   # change if you like
os.makedirs(DRIVE_ROOT + '/data', exist_ok=True)
os.makedirs(DRIVE_ROOT + '/results', exist_ok=True)
print('Persisting data/ and results/ under', DRIVE_ROOT)

Mounted at /content/drive
Persisting data/ and results/ under /content/drive/MyDrive/gnn-pruning


## 3. Clone the branch + install deps
No `torch_sparse` needed — the sparse path uses native `torch.sparse`.

In [3]:
%cd /content
![ -d GNN-Pruning-Research ] || git clone --branch main https://github.com/Mike-Mans/GNN-Pruning-Research.git
%cd /content/GNN-Pruning-Research
!git fetch origin && git checkout main && git pull --ff-only

/content
Cloning into 'GNN-Pruning-Research'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 227 (delta 81), reused 192 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (227/227), 398.78 KiB | 30.67 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/GNN-Pruning-Research
Already on 'main'
Your branch is up to date with 'origin/main'.
Already up to date.


In [4]:
# Symlink data/ and results/ to Drive so they persist across sessions.
import os, shutil
for d in ['data', 'results']:
    if os.path.islink(d):
        continue
    if os.path.exists(d):
        shutil.rmtree(d)
    os.symlink(f'{DRIVE_ROOT}/{d}', d)
!ls -la data results

lrwxrwxrwx 1 root root 39 Jun  2 13:23 data -> /content/drive/MyDrive/gnn-pruning/data
lrwxrwxrwx 1 root root 42 Jun  2 13:23 results -> /content/drive/MyDrive/gnn-pruning/results


In [5]:
# Colab ships torch+CUDA. Add PyG + project deps (no torch_sparse needed).
!pip -q install torch_geometric ogb rdkit
!pip -q install -e .
print('install done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 67.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gnn-pruning (pyproject.toml) ... done
install done


## 4. Run the sweep (no-pruning FIRST — pruning loads its checkpoints)
Large datasets are auto-ordered last. Expect a few hours on A100. `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` reduces fragmentation.

In [6]:
import os, subprocess, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

CONFIGS = {
    'no-pruning':      'src/gnn_pruning/configs/no_pruning.yaml',
    'magnitude':       'src/gnn_pruning/configs/magnitude.yaml',
    'wanda-uniform':   'src/gnn_pruning/configs/wanda_uniform.yaml',
    'wanda-degree':    'src/gnn_pruning/configs/wanda_degree.yaml',
    'wanda-per-class': 'src/gnn_pruning/configs/wanda_per_class.yaml',
}
for method, cfg in CONFIGS.items():
    print(f'\n===== {method} =====', flush=True)
    rc = subprocess.run(
        [sys.executable, '-m', 'gnn_pruning.cli', 'sweep',
         '--method', method, '--config', cfg],
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    ).returncode
    print(f'{method} finished (rc={rc})', flush=True)
print('\nSWEEP COMPLETE')


===== no-pruning =====
no-pruning finished (rc=0)

===== magnitude =====
magnitude finished (rc=0)

===== wanda-uniform =====
wanda-uniform finished (rc=0)

===== wanda-degree =====
wanda-degree finished (rc=0)

===== wanda-per-class =====
wanda-per-class finished (rc=0)

SWEEP COMPLETE


## 5. Inspect results

In [7]:
import pandas as pd, glob
for f in sorted(glob.glob('results/*/summary.csv')):
    df = pd.read_csv(f)
    print('\n=== ', f, ' (rows:', len(df), ') ===')
    display(df.head(12))

print('\n--- failures / OOMs across run logs (expect reddit/gat) ---')
!grep -H -A1 'FAILED' results/*/run.log | head -40


===  results/magnitude/summary.csv  (rows: 243 ) ===


,dataset,architecture,sparsity,metric_name,metric_value,n_runs
0,actor,gat,0.1,macro_f1,0.227338,10
1,actor,gat,0.2,macro_f1,0.226454,10
2,actor,gat,0.3,macro_f1,0.225089,10
3,actor,gat,0.4,macro_f1,0.226455,10
4,actor,gat,0.5,macro_f1,0.225813,10
5,actor,gat,0.6,macro_f1,0.225377,10
6,actor,gat,0.7,macro_f1,0.225435,10
7,actor,gat,0.8,macro_f1,0.220895,10
8,actor,gat,0.9,macro_f1,0.207815,10
9,actor,gcn,0.1,macro_f1,0.242040,10



===  results/no-pruning/summary.csv  (rows: 27 ) ===


,dataset,architecture,sparsity,metric_name,metric_value,n_runs
0,actor,gat,0.0,macro_f1,0.227184,10
1,actor,gcn,0.0,macro_f1,0.241879,10
2,bbbp,gcn,0.0,accuracy,0.800000,1
3,bbbp,graphsage,0.0,accuracy,0.780488,1
4,citeseer,gat,0.0,accuracy,0.678000,1
5,citeseer,gcn,0.0,accuracy,0.703000,1
6,computers,gat,0.0,accuracy,0.900036,1
7,computers,gcn,0.0,accuracy,0.905125,1
8,cora,gat,0.0,accuracy,0.800000,1
9,cora,gcn,0.0,accuracy,0.805000,1



===  results/wanda-degree/summary.csv  (rows: 243 ) ===


...


===  results/wanda-per-class/summary.csv  (rows: 243 ) ===


,dataset,architecture,sparsity,metric_name,metric_value,n_runs
0,actor,gat,0.1,macro_f1,0.226383,10
1,actor,gat,0.2,macro_f1,0.226626,10
2,actor,gat,0.3,macro_f1,0.226404,10
3,actor,gat,0.4,macro_f1,0.226883,10
4,actor,gat,0.5,macro_f1,0.226485,10
5,actor,gat,0.6,macro_f1,0.227074,10
6,actor,gat,0.7,macro_f1,0.222135,10
7,actor,gat,0.8,macro_f1,0.216554,10
8,actor,gat,0.9,macro_f1,0.198184,10
9,actor,gcn,0.1,macro_f1,0.242720,10



===  results/wanda-uniform/summary.csv  (rows: 243 ) ===


,dataset,architecture,sparsity,metric_name,metric_value,n_runs
0,actor,gat,0.1,macro_f1,0.226248,10
1,actor,gat,0.2,macro_f1,0.226321,10
2,actor,gat,0.3,macro_f1,0.226565,10
3,actor,gat,0.4,macro_f1,0.225440,10
4,actor,gat,0.5,macro_f1,0.226061,10
5,actor,gat,0.6,macro_f1,0.224742,10
6,actor,gat,0.7,macro_f1,0.222314,10
7,actor,gat,0.8,macro_f1,0.216511,10
8,actor,gat,0.9,macro_f1,0.196640,10
9,actor,gcn,0.1,macro_f1,0.242447,10



--- failures / OOMs across run logs (expect reddit/gat) ---
results/magnitude/run.log:  FAILED in 8.8s (rc=1)
results/magnitude/run.log-  stdout-tail:
--
results/magnitude/run.log:  FAILED in 8.8s (rc=1)
results/magnitude/run.log-  stdout-tail:
--
results/magnitude/run.log:  FAILED in 8.6s (rc=1)
results/magnitude/run.log-  stdout-tail:
--
results/magnitude/run.log:  FAILED in 8.8s (rc=1)
results/magnitude/run.log-  stdout-tail:
--
results/no-pruning/run.log:  FAILED in 13.1s (rc=1)
results/no-pruning/run.log-  stdout-tail:
--
results/no-pruning/run.log:  FAILED in 9.4s (rc=1)
results/no-pruning/run.log-  stdout-tail:
--
results/no-pruning/run.log:  FAILED in 36.9s (rc=1)
results/no-pruning/run.log-  stdout-tail:
--
results/no-pruning/run.log:  FAILED in 9.2s (rc=1)
results/no-pruning/run.log-  stdout-tail:
--
results/wanda-degree/run.log:  FAILED in 8.8s (rc=1)
results/wanda-degree/run.log-  stdout-tail:
--
results/wanda-degree/run.log:  FAILED in 8.8s (rc=1)
results/wanda-degree/run

## Notes
- **Resuming after a disconnect:** re-run cells 1–4. Idempotency skips every
  `(dataset, arch, seed, split)` already on Drive; only unfinished cells run.
- **Multi-seed pass:** edit each `src/gnn_pruning/configs/*.yaml` to
  `seeds: [0, 1, 2, 3, 4]`, push, and re-run cell 4. Finished seed-0 cells are
  skipped; seeds 1–4 fill in. (Large-NC × 5 is the slow part — run selectively.)
- **Outputs** live in `results/<method>/<dataset>/<arch>/seed-<N>/split-<M>/`
  on your Drive; `summary.csv` has the mean over seeds × splits with `n_runs`.
